# RAS 2026 — CG + AISO-Destroy + NDOR+SA

**Speed optimizations vs ras_ndor.ipynb:**
- od_matrix cache: `joblib` compress=3 → L3 load 36min → ~3min
- C++ parallel pricing (OpenMP) → CG pricing fast
- CG initial solution → better start → fewer NDOR episodes needed
- AISO destroy (Smart M on block-sharing) → diverse neighborhoods

**Pipeline:** CG → BC → AISO-NDOR+SA → Final SA

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CACHE = '/content/drive/MyDrive/ras2026_cache'
os.makedirs(DRIVE_CACHE, exist_ok=True)

if not os.path.exists('ras2026'):
    !git clone https://github.com/nepersoned/ras2026.git
os.chdir('ras2026')
print('cwd:', os.getcwd())

In [ ]:
!pip install -q pybind11 ortools scipy numpy pandas torch joblib

In [ ]:
# Drive 데이터 심볼릭 링크
DRIVE_DATA = '/content/drive/MyDrive/ras_release_v1.1'
if not os.path.exists('ras_release_v1.1'):
    os.symlink(DRIVE_DATA, 'ras_release_v1.1')
print('Data ready:', os.path.exists('ras_release_v1.1'))

## 1. Build C++ Modules

In [ ]:
import subprocess, sys

pybind_inc = subprocess.check_output(
    [sys.executable, '-m', 'pybind11', '--includes']
).decode().strip()

ext_suffix = subprocess.check_output(
    [sys.executable, '-c', 'import sysconfig; print(sysconfig.get_config_var("EXT_SUFFIX"))']
).decode().strip()

# evaluator (SA + O(1) delta)
out_ev = f'src/evaluator{ext_suffix}'
cmd_ev = f'g++ -O3 -shared -fPIC {pybind_inc} src/evaluator.cpp -o {out_ev}'
ret = os.system(cmd_ev)
print('evaluator:', 'OK' if ret == 0 else f'ERROR {ret}')

# cg_pricer (parallel pricing, OpenMP)
out_cg = f'src/cg_pricer{ext_suffix}'
cmd_cg = f'g++ -O3 -fopenmp -shared -fPIC {pybind_inc} src/cg_pricer.cpp -o {out_cg}'
ret = os.system(cmd_cg)
print('cg_pricer:', 'OK' if ret == 0 else f'ERROR {ret}')

In [ ]:
import sys
sys.path.insert(0, 'src')
sys.path.insert(0, '.')

import math, random, time, json, csv
from pathlib import Path
from copy import deepcopy
from collections import defaultdict, Counter

import numpy as np
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from scipy.stats import spearmanr

import evaluator as ev
import cg_pricer as cgp

from solver import (
    load_layer, load_od_matrix, Network, Demand, Solution,
    greedy_construct, evaluate, build_json,
    COMMODITY_TO_BLOCK_TYPE, DIRECT_ONLY,
)
from src.repair import ortools_repair

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 2. Environment + joblib Cache

In [ ]:
COMMODITY_IDX    = {'Manifest': 0, 'Bulk': 1, 'Intermodal': 2, 'Multilevel': 3}
DIRECT_ONLY_COMM = {'Intermodal', 'Automobile'}

MAX_HUBS = 50
FEAT_DIM = 7 + MAX_HUBS * 4   # 207
SEL_DIM  = FEAT_DIM + 3       # 210


def load_env(layer, multiplier):
    """Load environment with joblib-compressed od_matrix cache."""
    # joblib cache path (much faster than pickle for large dicts)
    cache_jl  = f'{DRIVE_CACHE}/od_matrix_{layer}.joblib'
    cache_pkl = f'{DRIVE_CACHE}/od_matrix_{layer}.pkl'  # old format fallback

    nodes_df, links_df, demands_scaled, demands_raw, settings = load_layer(layer, multiplier)
    yard_rows = nodes_df[nodes_df['node_type'] == 'yard']
    yard_info = {
        int(r['node_id']): {
            'num_tracks':        float(r.get('num_tracks', 9999) or 9999),
            'handling_capacity': float(r.get('handling_capacity', 1e9) or 1e9),
            'handling_cost':     float(r.get('handling_cost', 0) or 0),
        } for _, r in yard_rows.iterrows()
    }
    origin_ids   = set(demands_scaled['origin_yard_id'].astype(int))
    dest_ids     = set(demands_scaled['dest_yard_id'].astype(int))
    all_yard_ids = sorted(origin_ids | dest_ids)

    # ── joblib cache (compress=3: ~5x faster than pickle for large objects) ────
    if os.path.exists(cache_jl):
        t0 = time.time()
        print(f'  Loading od_matrix from joblib cache: {cache_jl}')
        od_matrix = joblib.load(cache_jl)
        print(f'  Loaded in {time.time()-t0:.1f}s')
    elif os.path.exists(cache_pkl):
        # Migrate old pickle → joblib
        import pickle
        print(f'  Migrating old pickle → joblib cache...')
        t0 = time.time()
        with open(cache_pkl, 'rb') as f:
            od_matrix = pickle.load(f)
        print(f'  Pickle loaded in {time.time()-t0:.1f}s, saving as joblib...')
        joblib.dump(od_matrix, cache_jl, compress=3)
        print(f'  Saved joblib cache (future loads will be ~5x faster)')
    else:
        print(f'  Computing od_matrix (will save as joblib)...')
        all_od_pairs = {(o,d) for o in all_yard_ids for d in all_yard_ids if o!=d}
        od_matrix    = load_od_matrix(all_od_pairs)
        joblib.dump(od_matrix, cache_jl, compress=3)
        print(f'  Saved to {cache_jl}')

    net = Network(nodes_df, links_df, origin_ids, dest_ids, settings, verbose=True)
    demands = [
        Demand(
            idx=idx, demand_id=int(r['demand_id']),
            commodity_type=str(r['block_type']),
            block_type=COMMODITY_TO_BLOCK_TYPE.get(str(r['block_type']), 'Manifest'),
            origin=int(r['origin_yard_id']), dest=int(r['dest_yard_id']),
            volume=int(r['volume']),
            sp_dist=od_matrix.get(
                (int(r['origin_yard_id']), int(r['dest_yard_id'])),
                net.dist(int(r['origin_yard_id']), int(r['dest_yard_id']))
            ),
        ) for idx, (_, r) in enumerate(demands_scaled.iterrows())
    ]
    return dict(net=net, od_matrix=od_matrix, settings=settings,
                yard_info=yard_info, demands=demands, all_yards=all_yard_ids,
                nodes_df=nodes_df, links_df=links_df, demands_raw=demands_raw)


def ranked_hubs(dem, env):
    if dem.commodity_type in DIRECT_ONLY_COMM:
        return []
    scored = []
    for hub in env['all_yards']:
        if hub in (dem.origin, dem.dest): continue
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2): continue
        scored.append((d1+d2, hub))
    scored.sort()
    return [h for _,h in scored]


def demand_feat(dem, hubs, env):
    od_d  = env['net'].dist(dem.origin, dem.dest)
    od_ic = env['net'].interchanges(dem.origin, dem.dest)
    bt_oh = [1.0 if COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest')==t else 0.0
             for t in ['Manifest','Bulk','Intermodal','Multilevel']]
    base = [math.log1p(dem.volume),
            math.log1p(od_d if math.isfinite(od_d) else 1e6),
            od_ic/5.0] + bt_oh
    hub_f = [0.0]*(MAX_HUBS*4)
    for j, hub in enumerate(hubs[:MAX_HUBS]):
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2): continue
        ic1 = env['net'].interchanges(dem.origin, hub)
        ic2 = env['net'].interchanges(hub, dem.dest)
        hc  = env['yard_info'].get(hub,{}).get('handling_cost',0.0)
        hub_f[j*4]  = math.log1p(d1)
        hub_f[j*4+1]= math.log1p(d2)
        hub_f[j*4+2]= (ic1+ic2)/5.0
        hub_f[j*4+3]= hc/500.0
    return base + hub_f


def _build_cands(dem, hubs, env):
    s  = env['settings']
    tc = float(s.get('transport_cost_coefficient', 1.0))
    ic = float(s.get('interchange_cost', 100))
    M  = float(s.get('stress_penalty_M', 5))
    net= env['net']
    cands = []
    cands.append((None, M * dem.volume * dem.sp_dist))
    od_d = net.dist(dem.origin, dem.dest)
    if not math.isinf(od_d):
        od_ic = net.interchanges(dem.origin, dem.dest)
        cands.append(([(dem.origin, dem.dest)],
                      tc*dem.volume*od_d + ic*dem.volume*od_ic))
    if dem.commodity_type not in DIRECT_ONLY_COMM:
        for hub in hubs[:MAX_HUBS]:
            d1 = net.dist(dem.origin, hub)
            d2 = net.dist(hub, dem.dest)
            if math.isinf(d1) or math.isinf(d2): continue
            ic1= net.interchanges(dem.origin, hub)
            ic2= net.interchanges(hub, dem.dest)
            hc = env['yard_info'].get(hub,{}).get('handling_cost',0.0)
            cands.append(([(dem.origin,hub),(hub,dem.dest)],
                          tc*dem.volume*(d1+d2) + ic*dem.volume*(ic1+ic2) + hc*dem.volume))
    return cands


def precompute(env):
    data = []
    for dem in env['demands']:
        if dem.volume <= 0:
            data.append(None); continue
        hubs  = ranked_hubs(dem, env)
        feat  = demand_feat(dem, hubs, env)
        cands = _build_cands(dem, hubs, env)
        data.append({'feat': feat, 'hubs': hubs, 'candidates': cands,
                     'direct_only': dem.commodity_type in DIRECT_ONLY_COMM})
    return data


print('Environment + joblib cache ready.')

## 3. Column Generation (CG) Initial Solution

In [ ]:
from ortools.linear_solver import pywraplp


def cg_solve(env, dd, max_iter=30, time_limit_s=120, verbose=True):
    """
    Column Generation for railroad blocking initial solution.

    Master LP: min Σ cost[i][r] * x[i][r]
               s.t. Σ_r x[i][r] = 1 for each demand i
                    x[i][r] >= 0

    Pricing: for each demand i, find route r* with
             reduced_cost = cost[i][r*] - dual[i] < 0
             via C++ parallel scan over precomputed candidates.

    Returns: routings (list of routes, one per demand)
    """
    demands = env['demands']
    n = len(demands)
    t_start = time.time()

    # Build candidate cost table for C++ pricer
    # cand_costs[i] = [(route_idx, cost), ...]
    cand_costs = []
    for i, d in enumerate(dd):
        if d is None:
            cand_costs.append([])
        else:
            cand_costs.append([(ri, cost) for ri, (route, cost) in enumerate(d['candidates'])])

    # Initial columns: greedy (cheapest candidate per demand)
    # active_cols[i] = set of route indices currently in master LP
    active_cols = []
    for i, d in enumerate(dd):
        if d is None:
            active_cols.append({0})
        else:
            # Start with cheapest candidate
            best_ri = min(range(len(d['candidates'])), key=lambda r: d['candidates'][r][1])
            active_cols.append({best_ri})

    best_routings = None
    best_obj = float('inf')

    for it in range(max_iter):
        if time.time() - t_start > time_limit_s:
            if verbose: print(f'  CG: time limit {time_limit_s}s reached at iter {it}')
            break

        # ── Solve master LP ──────────────────────────────────────────────────
        solver = pywraplp.Solver.CreateSolver('GLOP')
        solver.SuppressOutput()

        # Variables: x[i][ri]
        x = {}
        for i in range(n):
            for ri in active_cols[i]:
                x[(i, ri)] = solver.NumVar(0.0, 1.0, f'x_{i}_{ri}')

        # Constraints: Σ_ri x[i][ri] = 1 for each i
        for i in range(n):
            ct = solver.Constraint(1.0, 1.0, f'dem_{i}')
            for ri in active_cols[i]:
                ct.SetCoefficient(x[(i, ri)], 1.0)

        # Objective: min Σ cost * x
        obj = solver.Objective()
        obj.SetMinimization()
        for (i, ri), var in x.items():
            d = dd[i]
            cost = d['candidates'][ri][1] if d else 0.0
            obj.SetCoefficient(var, cost)

        status = solver.Solve()
        if status not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
            if verbose: print(f'  CG iter {it}: LP infeasible, stopping')
            break

        lp_obj = solver.Objective().Value()

        # ── Extract dual variables ────────────────────────────────────────────
        duals = [0.0] * n
        for i in range(n):
            ct = solver.LookupConstraint(f'dem_{i}')
            if ct:
                duals[i] = ct.dual_value()

        # ── Pricing: C++ parallel find negative reduced cost columns ──────────
        new_cols = cgp.price_all(cand_costs, duals, tol=1e-6)

        added = 0
        for pr in new_cols:
            if pr.route_idx not in active_cols[pr.demand_idx]:
                active_cols[pr.demand_idx].add(pr.route_idx)
                added += 1

        # Round LP solution → integer routes
        routings = []
        for i in range(n):
            d = dd[i]
            if d is None:
                routings.append(None); continue
            best_ri = max(active_cols[i], key=lambda ri: x[(i,ri)].solution_value())
            routings.append(d['candidates'][best_ri][0])

        if lp_obj < best_obj:
            best_obj = lp_obj
            best_routings = routings

        if verbose:
            print(f'  CG iter {it+1:3d} | LP obj={lp_obj:>18,.0f} | '
                  f'new cols={added:5d} | elapsed={time.time()-t_start:.1f}s')

        if added == 0:
            if verbose: print(f'  CG converged after {it+1} iterations.')
            break

    if best_routings is None:
        # Fallback to greedy
        sol = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                               env['settings'], env['yard_info'])
        best_routings = sol.routings

    return best_routings


print('Column Generation ready.')

## 4. AISO Destroy with Smart M

In [ ]:
"""
AISO Destroy: replaces RL SelectionPolicy with AISO agents.

Each demand = one potential destroy target.
AISO agents carry type vectors W_i ∈ Δ^{K-1} over K demand clusters.
Smart M encodes:
  - Block-sharing affinity: demands on same segment → destroy together
  - Volume-weighted MI: high-volume demands guide low-volume agents
  - Asymmetric: type k seeks type l if l has higher block competition

Output: probability distribution over demands → sample K to destroy.
"""

def build_demand_clusters(dd, env, K=10):
    """
    Cluster demands by routing similarity (shared segments).
    Returns cluster assignments and per-cluster 'MI' (volume × competition).
    """
    from sklearn.cluster import AgglomerativeClustering
    demands = env['demands']
    n = len(demands)

    # Feature: [origin_norm, dest_norm, volume_norm, direct_only]
    all_yards = sorted(env['all_yards'])
    yard_to_idx = {y: i for i, y in enumerate(all_yards)}
    Y = len(all_yards)

    feats = []
    valid_idx = []
    for i, d in enumerate(dd):
        if d is None: continue
        dem = demands[i]
        feats.append([
            yard_to_idx.get(dem.origin, 0) / max(Y, 1),
            yard_to_idx.get(dem.dest, 0)   / max(Y, 1),
            math.log1p(dem.volume) / 15.0,
            1.0 if dem.commodity_type in DIRECT_ONLY_COMM else 0.0,
        ])
        valid_idx.append(i)

    if len(feats) < K:
        K = max(2, len(feats) // 2)

    X = np.array(feats, dtype=np.float32)
    clust = AgglomerativeClustering(n_clusters=K, linkage='ward').fit(X)
    labels = clust.labels_  # (len(valid_idx),)

    # Per-cluster 'MI' = mean volume (proxy for importance)
    cluster_vol = defaultdict(list)
    for j, i in enumerate(valid_idx):
        cluster_vol[labels[j]].append(demands[i].volume)
    mu = np.array([np.mean(cluster_vol.get(k, [1.0])) for k in range(K)])

    # Full assignment array
    cluster_of = [-1] * n
    for j, i in enumerate(valid_idx):
        cluster_of[i] = int(labels[j])

    return cluster_of, mu, K


def build_smart_M(mu, K, gamma=0.5):
    """
    Smart M for demand clustering:
      M[i][j] = -corr_penalty + gamma*(mu_tilde[j] - mu_tilde[i])
    Here corr_penalty = 0 (demands don't have feature correlation like AISO paper).
    The antisymmetric gradient alone creates directed routing: low-mu clusters
    are attracted to high-mu clusters → diverse destroy across volume levels.
    """
    mu_min, mu_max = mu.min(), mu.max()
    if mu_max - mu_min < 1e-8:
        return np.zeros((K, K))
    mu_norm = (mu - mu_min) / (mu_max - mu_min)

    M = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            if i != j:
                # antisymmetric information gradient (from AISO paper)
                M[i, j] = gamma * (mu_norm[j] - mu_norm[i])
    return M


class AISODestroy:
    """
    AISO-based destroy operator.

    N_agents agents carry type vectors W ∈ Δ^{K-1}.
    Compatibility: c_ij = W_i @ M @ W_j (asymmetric).
    Type assimilation on improvement.
    Adaptive repulsion when diversity drops.

    select(K_destroy, route_scores) → indices of K demands to destroy.
    route_scores[i] = current cost of demand i's route (higher = more to gain).
    """

    def __init__(self, cluster_of, mu, K_clusters,
                 N_agents=20, T_iters=30, alpha=0.4, beta=0.1, gamma=0.5):
        self.cluster_of  = cluster_of   # demand_idx → cluster
        self.K           = K_clusters
        self.N           = N_agents
        self.T           = T_iters
        self.alpha       = alpha
        self.beta        = beta
        self.M           = build_smart_M(mu, K_clusters, gamma)  # (K,K)

        # Uniform init on simplex
        W = np.random.dirichlet(np.ones(K_clusters), size=N_agents)
        self.W = W  # (N, K)

    def _compat(self, i, j):
        return float(self.W[i] @ self.M @ self.W[j])

    def _diversity(self):
        # Mean pairwise L2 distance in type space (normalized)
        dists = []
        for i in range(self.N):
            for j in range(i+1, self.N):
                dists.append(np.linalg.norm(self.W[i] - self.W[j]))
        return np.mean(dists) if dists else 1.0

    def select(self, K_destroy, route_costs, valid_indices):
        """
        Run AISO for T iterations, return K_destroy demand indices to destroy.

        route_costs: array of current costs per demand (higher = priority)
        valid_indices: list of valid demand indices
        """
        if len(valid_indices) == 0:
            return []

        K  = self.K
        N  = self.N
        rc = np.array(route_costs, dtype=np.float64)
        rc_max = rc.max() if rc.max() > 0 else 1.0

        # Agent positions: index into valid_indices
        # (agents move through demand-index space by adjusting their cluster focus)
        best_scores = np.full(N, -np.inf)
        # Score of agent i = route_cost of the demand whose cluster best matches W[i]

        # Per-cluster demand lists for efficient selection
        cluster_demands = defaultdict(list)
        for vi in valid_indices:
            c = self.cluster_of[vi]
            if c >= 0:
                cluster_demands[c].append(vi)

        # Agent → current focused demand (initially random)
        agent_demand = [
            valid_indices[np.random.randint(len(valid_indices))]
            for _ in range(N)
        ]

        for t in range(self.T):
            delta = self._diversity()
            # Adaptive repulsion multiplier
            repul = 1.0 + 3.0 * math.exp(-delta / 0.12)

            M_eff = self.M.copy()
            M_eff[M_eff < 0] *= repul

            for i in range(N):
                # Find best partner
                s_i = rc[agent_demand[i]] / rc_max
                best_c = -np.inf
                best_j = -1
                for j in range(N):
                    if j == i: continue
                    s_j = rc[agent_demand[j]] / rc_max
                    c_ij = float(self.W[i] @ M_eff @ self.W[j]) * s_j
                    if c_ij > best_c:
                        best_c = c_ij
                        best_j = j

                if best_j < 0: continue

                # Move toward partner's cluster focus
                # Simulate position update in cluster-weight space
                new_W = self.W[i] + self.alpha * best_c * (self.W[best_j] - self.W[i])
                new_W = np.clip(new_W, 1e-8, None)
                new_W /= new_W.sum()

                # Find demand in the most-weighted cluster for new_W
                top_cluster = int(np.argmax(new_W))
                cands_in_cluster = cluster_demands.get(top_cluster, valid_indices)
                new_demand = cands_in_cluster[np.random.randint(len(cands_in_cluster))]

                new_score = rc[new_demand] / rc_max
                old_score = rc[agent_demand[i]] / rc_max

                if new_score >= old_score:
                    # Accept: update position + type assimilation
                    self.W[i] = new_W
                    agent_demand[i] = new_demand
                    best_scores[i] = new_score

                    # Assimilate type toward partner
                    mixed = (1 - self.beta) * self.W[i] + self.beta * self.W[best_j]
                    mixed /= mixed.sum()
                    self.W[i] = mixed

        # Aggregate: cluster selection weights = mean W across agents
        cluster_weight = self.W.mean(axis=0)  # (K,)

        # Score each valid demand: cluster_weight[cluster] * route_cost
        demand_scores = {}
        for vi in valid_indices:
            c = self.cluster_of[vi]
            w = cluster_weight[c] if c >= 0 else 1.0 / max(K, 1)
            demand_scores[vi] = w * rc[vi]

        # Top-K by combined score
        sorted_demands = sorted(demand_scores, key=demand_scores.get, reverse=True)
        return sorted_demands[:K_destroy]


print('AISO destroy operator ready.')

## 5. C++ Solver Setup

In [ ]:
def build_cpp_solver(env, dd, init_routings, verbose=False):
    demands = env['demands']
    s = env['settings']

    dem_list = []
    for i, dem in enumerate(demands):
        comm_idx = COMMODITY_IDX.get(COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest'), 0)
        dem_list.append((
            i, dem.origin, dem.dest, dem.volume,
            dem.sp_dist, comm_idx,
            dem.commodity_type in DIRECT_ONLY_COMM
        ))

    cand_list = []
    for i, d in enumerate(dd):
        if d is None:
            cand_list.append([(True, False, -1, 0.0, 0.0, -1, -1, -1, -1)])
            continue
        dem   = demands[i]
        cands = d['candidates']
        cpp_cands = []
        for route, cost in cands:
            if route is None:
                cpp_cands.append((True, False, -1, cost, 0.0, -1, -1, -1, -1))
            elif len(route) == 1:
                o, dst = route[0]
                cpp_cands.append((False, True, -1, cost, 0.0, o, dst, -1, -1))
            else:
                (o1,h),(h2,d2) = route
                hc = env['yard_info'].get(h,{}).get('handling_cost',0.0) * dem.volume
                cpp_cands.append((False, False, h, cost-hc, hc, o1, h, h, d2))
        cand_list.append(cpp_cands)

    init_route_idx = []
    for i, routing in enumerate(init_routings):
        d = dd[i]
        if d is None:
            init_route_idx.append(0); continue
        found = 0
        for ri, (route, _) in enumerate(d['candidates']):
            if route == routing:
                found = ri; break
        init_route_idx.append(found)

    yd_tracks = {k: int(v['num_tracks']) for k,v in env['yard_info'].items()}
    yd_hcap   = {k: float(v['handling_capacity']) for k,v in env['yard_info'].items()}
    seg_dist  = {f'{o}_{d}': float(dist)
                 for (o,d), dist in env['od_matrix'].items()
                 if math.isfinite(dist)}

    cpp_settings = ev.Settings()
    cpp_settings.block_fixed_cost     = float(s.get('block_fixed_cost', 1500))
    cpp_settings.unserved_penalty     = float(s.get('stress_penalty_M', 5))
    cpp_settings.min_block_vol_short  = 5.0
    cpp_settings.min_block_vol_medium = 10.0
    cpp_settings.min_block_vol_long   = 15.0

    solver = ev.RasSolver()
    solver.init(dem_list, cand_list, init_route_idx,
                yd_tracks, yd_hcap, seg_dist, cpp_settings)

    if verbose:
        print(f'C++ solver init: score={solver.get_score():,.0f}')
    return solver, cand_list


def route_idx_to_routing(route_indices, dd, env):
    routings = []
    for i, ri in enumerate(route_indices):
        d = dd[i]
        if d is None:
            routings.append(None); continue
        route, _ = d['candidates'][ri]
        routings.append(route)
    return routings


print('C++ solver setup ready.')

## 6. CG-AISO-NDOR+SA Training Loop

In [ ]:
def cg_aiso_ndor_train(
    env, dd,
    n_episodes=150,
    sa_iters_per_ep=5000,
    T0_frac=0.05,
    T_final_frac=0.001,
    K_destroy=10,
    K_clusters=10,
    aiso_agents=20,
    aiso_iters=30,
    final_sa_iters=200000,
    cg_max_iter=20,
    cg_time_limit=90,
    print_every=25,
):
    t_total = time.time()

    # ── Step 1: CG initial solution ──────────────────────────────────────────
    print('\n── CG ──')
    cg_routings = cg_solve(env, dd, max_iter=cg_max_iter,
                           time_limit_s=cg_time_limit, verbose=True)

    cpp_solver, _ = build_cpp_solver(env, dd, cg_routings, verbose=True)
    cg_score = cpp_solver.get_score()
    print(f'  CG score: {cg_score:,.0f}')

    # Compare to greedy
    greedy_sol = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                                  env['settings'], env['yard_info'])
    greedy_score, _ = evaluate(greedy_sol, env['net'], env['od_matrix'],
                               env['settings'], env['yard_info'])
    if cg_score > greedy_score:
        print(f'  CG worse than greedy ({greedy_score:,.0f}), using greedy')
        cpp_solver, _ = build_cpp_solver(env, dd, greedy_sol.routings, verbose=False)
        cg_score = cpp_solver.get_score()

    init_score = cg_score

    # ── Step 2: Build AISO destroy operator ─────────────────────────────────
    print('\n── Building AISO Smart M ──')
    cluster_of, mu, K_used = build_demand_clusters(dd, env, K=K_clusters)
    print(f'  {K_used} demand clusters, mu range [{mu.min():.1f}, {mu.max():.1f}]')
    aiso = AISODestroy(cluster_of, mu, K_used,
                       N_agents=aiso_agents, T_iters=aiso_iters)

    # ── Step 3: NDOR+SA with AISO destroy ────────────────────────────────────
    best_score  = cpp_solver.get_score()
    best_routes = list(cpp_solver.get_routes())
    T0          = best_score * T0_frac
    T_final     = best_score * T_final_frac

    log = []
    print(f'\n── AISO-NDOR+SA ──  init={best_score:,.0f}  episodes={n_episodes}')
    print(f'  SA: {sa_iters_per_ep} iters/ep  OR-Tools: K={K_destroy}  '
          f'AISO: {aiso_agents} agents/{aiso_iters} iters')

    for ep in range(1, n_episodes+1):
        frac = ep / n_episodes
        T_ep = T0 * ((T_final / T0) ** frac)

        # AISO destroy: score demands by current route cost
        route_idx    = list(cpp_solver.get_routes())
        route_costs  = np.zeros(len(dd))
        valid_indices = []
        for i, d in enumerate(dd):
            if d is None: continue
            ri = route_idx[i]
            _, cost = d['candidates'][ri]
            route_costs[i] = cost
            valid_indices.append(i)

        top_k = aiso.select(K_destroy, route_costs, valid_indices)

        # Build RL-style weights for C++ SA from AISO selection
        weights_full = [1e-6] * len(dd)
        for i in top_k:
            weights_full[i] = 1.0 / max(len(top_k), 1)

        # C++ SA step
        score_before = cpp_solver.get_score()
        sa_result = cpp_solver.sa_run(
            weights_full,
            float(T_ep),
            float(T_ep * 0.01),
            int(sa_iters_per_ep),
            int(sa_iters_per_ep // 10),
            int(ep),
        )
        cpp_solver.set_routes(sa_result.best_routes)

        # OR-Tools repair on AISO-selected demands
        if top_k:
            current_routings = route_idx_to_routing(list(cpp_solver.get_routes()), dd, env)
            repaired = ortools_repair(top_k, dd, env, current_routings)
            cpp_solver2, _ = build_cpp_solver(env, dd, repaired, verbose=False)
            if cpp_solver2.get_score() < cpp_solver.get_score():
                cpp_solver = cpp_solver2

        score_after = cpp_solver.get_score()
        if score_after < best_score:
            best_score  = score_after
            best_routes = list(cpp_solver.get_routes())

        log.append({'ep': ep, 'score': score_after, 'best': best_score})

        if ep % print_every == 0:
            print(f'  ep {ep:4d} | score={score_after:>16,.0f} | best={best_score:>16,.0f}')

    # ── Step 4: Final SA polishing ────────────────────────────────────────────
    print(f'\n── Final SA polishing ({final_sa_iters:,} iters) ──')
    cpp_solver.set_routes(best_routes)
    uniform_w  = [1.0 / max(len(dd), 1)] * len(dd)
    T_pol_0    = best_score * 0.10
    T_pol_end  = best_score * 0.0001

    pol = cpp_solver.sa_run(
        uniform_w,
        float(T_pol_0), float(T_pol_end),
        int(final_sa_iters), int(final_sa_iters // 10),
        999,
    )
    if pol.best_score < best_score:
        best_score  = pol.best_score
        best_routes = list(pol.best_routes)
        print(f'  SA improved: {best_score:,.0f}')
    else:
        print(f'  SA no improvement. best={best_score:,.0f}')

    cpp_solver.set_routes(best_routes)
    best_routings = route_idx_to_routing(best_routes, dd, env)
    print(f'  Final best (C++): {best_score:,.0f}')
    print(f'  Total elapsed: {time.time()-t_total:.1f}s')
    return best_routings, best_score, log


print('CG-AISO-NDOR+SA loop ready.')

## 7. Train: L1 → L2 → L3

In [ ]:
CASE_ORDER = [
    ('l1', 0.5), ('l1', 1.0), ('l1', 2.0),
    ('l2', 0.5), ('l2', 1.0), ('l2', 2.0),
    ('l3', 0.5), ('l3', 1.0), ('l3', 2.0),
]

# Fewer episodes vs ras_ndor because CG gives better start
NDOR_EPISODES   = {'l1': 150,  'l2': 80,    'l3': 40}
SA_ITERS_PER_EP = {'l1': 5000, 'l2': 10000, 'l3': 20000}
K_DESTROY       = {'l1': 5,    'l2': 20,    'l3': 50}
K_CLUSTERS      = {'l1': 8,    'l2': 10,    'l3': 15}
FINAL_SA_ITERS  = {'l1': 200000, 'l2': 500000, 'l3': 1000000}
CG_MAX_ITER     = {'l1': 30,  'l2': 20, 'l3': 10}  # L3 fewer CG iters (154k demands)
CG_TIME_LIMIT   = {'l1': 60,  'l2': 90, 'l3': 120}

solutions  = {}
train_logs = {}
print('Hyperparameters set.')

In [ ]:
for mult in [0.5, 1.0, 2.0]:
    tag = f'l1_{mult}'
    print(f'\n{"="*60}\n  L1 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l1', mult)
    dd  = precompute(env)

    best_routings, best_score, log = cg_aiso_ndor_train(
        env, dd,
        n_episodes=NDOR_EPISODES['l1'],
        sa_iters_per_ep=SA_ITERS_PER_EP['l1'],
        K_destroy=K_DESTROY['l1'],
        K_clusters=K_CLUSTERS['l1'],
        final_sa_iters=FINAL_SA_ITERS['l1'],
        cg_max_iter=CG_MAX_ITER['l1'],
        cg_time_limit=CG_TIME_LIMIT['l1'],
    )
    train_logs[tag] = log

    sol = Solution(env['demands'], best_routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    print(f'  Elapsed: {time.time()-t0:.1f}s')

In [ ]:
for mult in [0.5, 1.0, 2.0]:
    tag = f'l2_{mult}'
    print(f'\n{"="*60}\n  L2 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l2', mult)
    dd  = precompute(env)

    best_routings, best_score, log = cg_aiso_ndor_train(
        env, dd,
        n_episodes=NDOR_EPISODES['l2'],
        sa_iters_per_ep=SA_ITERS_PER_EP['l2'],
        K_destroy=K_DESTROY['l2'],
        K_clusters=K_CLUSTERS['l2'],
        final_sa_iters=FINAL_SA_ITERS['l2'],
        cg_max_iter=CG_MAX_ITER['l2'],
        cg_time_limit=CG_TIME_LIMIT['l2'],
    )
    train_logs[tag] = log

    sol = Solution(env['demands'], best_routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    print(f'  Elapsed: {time.time()-t0:.1f}s')

In [ ]:
for mult in [0.5, 1.0, 2.0]:
    tag = f'l3_{mult}'
    print(f'\n{"="*60}\n  L3 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l3', mult)
    dd  = precompute(env)

    best_routings, best_score, log = cg_aiso_ndor_train(
        env, dd,
        n_episodes=NDOR_EPISODES['l3'],
        sa_iters_per_ep=SA_ITERS_PER_EP['l3'],
        K_destroy=K_DESTROY['l3'],
        K_clusters=K_CLUSTERS['l3'],
        final_sa_iters=FINAL_SA_ITERS['l3'],
        cg_max_iter=CG_MAX_ITER['l3'],
        cg_time_limit=CG_TIME_LIMIT['l3'],
    )
    train_logs[tag] = log

    sol = Solution(env['demands'], best_routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    print(f'  Elapsed: {time.time()-t0:.1f}s')

## 8. Generate submission.csv

In [ ]:
rows = [['ID', 'data']]
for case_id, (layer, mult) in enumerate(CASE_ORDER):
    tag = f'{layer}_{mult}'
    sol = solutions.get(tag)
    if sol is None:
        print(f'[ID {case_id}] {tag} MISSING')
        rows.append([case_id, '{}']); continue
    data_str = json.dumps(sol, separators=(',', ':'))
    print(f'[ID {case_id}] {tag}  blocks={len(sol["outputs"]["1 Block Design"])}  len={len(data_str):,}')
    rows.append([case_id, data_str])

with open('submission_cg.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(rows)
print(f'Done. {sum(len(r[1]) for r in rows[1:])/1e6:.1f} MB')

from google.colab import files
files.download('submission_cg.csv')

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (layer, mult) in zip(axes, [('l1',1.0),('l2',1.0),('l3',1.0)]):
    log = train_logs.get(f'{layer}_{mult}', [])
    if not log: continue
    ax.plot([d['ep'] for d in log], [d['best'] for d in log])
    ax.set_title(f'{layer.upper()} ×{mult}')
    ax.set_xlabel('Episode'); ax.set_ylabel('Best Stress Score')
    ax.grid(True, alpha=0.3)
plt.suptitle('CG + AISO-NDOR+SA Training Curves')
plt.tight_layout()
plt.savefig('training_curves_cg.png', dpi=150)
plt.show()